## PCA Regression


In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from scipy.sparse import issparse


In [ ]:

DATASET_ID = 3192  
PCA_VARIANCE = 0.5 

print(f"Fetching OpenML dataset {DATASET_ID}")
dataset = fetch_openml(data_id=DATASET_ID, as_frame=False, parser='auto')
X_raw = dataset.data
y_raw = dataset.target

# Handle Sparse
if issparse(X_raw):
    X_dense = pd.DataFrame(X_raw.toarray())
else:
    X_dense = pd.DataFrame(X_raw).select_dtypes(include=[np.number])


In [ ]:

# Impute
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X_dense)

# Auto-Log (Check for negatives)
if np.min(X_imputed) >= 0:
    print("Applying Log2(x+1) (Positive Data)")
    X_processed = np.log2(X_imputed + 1)
else:
    print("Skipping Log (Contains Negatives)")
    X_processed = X_imputed

# SCale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_processed)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_raw, test_size=0.2, random_state=42
)

print(f" Training set shape: {X_train.shape}")


In [ ]:

print("\n-Baseline")
rf_base = RandomForestRegressor(n_estimators=100, random_state=42)
rf_base.fit(X_train, y_train)

y_pred_base = rf_base.predict(X_test)
r2_base = r2_score(y_test, y_pred_base)
mse_base = mean_squared_error(y_test, y_pred_base)

print(f"Baseline R2:  {r2_base:.4f} (Accuracy)")
print(f"Baseline MSE: {mse_base:.4f} (Error)")



In [ ]:

print("\n PCA ")

pca = PCA(n_components=PCA_VARIANCE, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

n_comps = X_train_pca.shape[1]
print(f"PCA reduced features from {X_train.shape[1]} -> {n_comps} components.")

rf_pca = RandomForestRegressor(n_estimators=100, random_state=42)
rf_pca.fit(X_train_pca, y_train)

y_pred_pca = rf_pca.predict(X_test_pca)
r2_pca = r2_score(y_test, y_pred_pca)
mse_pca = mean_squared_error(y_test, y_pred_pca)


In [ ]:

print(f"PCA R2:       {r2_pca:.4f}")
print(f"PCA MSE:      {mse_pca:.4f}")
print(f"Baseline R2: {r2_base:.4f}")
print(f"PCA R2:      {r2_pca:.4f}")


# PCA on CSV

In [ ]:

FILE_PATH = "datasets/df_destx.csv"  
TARGET_COLUMN = None      
PCA_VARIANCE = 0.7       

df = pd.read_csv(FILE_PATH)

print(f"Raw Shape: {df.shape}")


In [ ]:

# Separate Target (y) and Features (X)
if TARGET_COLUMN not in df.columns:
    print(f" Error: Target column '{TARGET_COLUMN}' not found in CSV.")
    print(f" Available columns: {list(df.columns)}")
if TARGET_COLUMN is None:
    TARGET_COLUMN = df.columns[-1]  # Auto-select last column as target
    print(f"Target not set. Auto-selected last column: '{TARGET_COLUMN}'")

y = df[TARGET_COLUMN].values
X_raw = df.drop(columns=[TARGET_COLUMN])


In [ ]:

# Drop non-numeric columns (ID, Name, Date, etc.)
X_numeric = X_raw.select_dtypes(include=[np.number])
dropped_cols = [c for c in X_raw.columns if c not in X_numeric.columns]

if len(dropped_cols) > 0:
    print(f"Dropped {len(dropped_cols)} non-numeric columns (e.g., {dropped_cols[:3]}...)")
print(" Preprocessing...")

# Impute Missing Values (Mean)
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X_numeric)

# Auto-Log Transform (Only if data is strictly positive)
# This helps normalize skewed data like prices or counts
if np.min(X_imputed) >= 0:
    print("Applying Log2(x+1) transformation...")
    X_processed = np.log2(X_imputed + 1)
else:
    print("Skipping Log (Contains negative values)...")
    X_processed = X_imputed


In [ ]:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_processed)

# Split Train/Test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"Training Data: {X_train.shape}")
print(f"Testing Data:  {X_test.shape}")


In [ ]:
print("\n Baseline: Random Forest (All Features) ")
rf_base = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_base.fit(X_train, y_train)

y_pred_base = rf_base.predict(X_test)

r2_base = r2_score(y_test, y_pred_base)
mse_base = mean_squared_error(y_test, y_pred_base)
print(f"   R2 Score: {r2_base:.4f}")
print(f"   MSE:      {mse_base:.4f}")


In [ ]:
print(f"\n PCA (Variance={PCA_VARIANCE}) ")

# Fit PCA on Train
pca = PCA(n_components=PCA_VARIANCE, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)
n_comps = X_train_pca.shape[1]
original_feats = X_train.shape[1]
print(f"   Compressed features: {original_feats} -> {n_comps}")

# Train RF on Compressed Data
rf_pca = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_pca.fit(X_train_pca, y_train)

# Predict
y_pred_pca = rf_pca.predict(X_test_pca)
r2_pca = r2_score(y_test, y_pred_pca)
mse_pca = mean_squared_error(y_test, y_pred_pca)


In [ ]:
print(f"   R2 Score: {r2_pca:.4f}")
print(f"   MSE:      {mse_pca:.4f}")
print(f"{'METRIC':<10} | {'BASELINE':<10} | {'PCA':<10}")
print(f"{'R2 Score':<10} | {r2_base:.4f}     | {r2_pca:.4f}")
print(f"{'MSE Error':<10} | {mse_base:.4f}     | {mse_pca:.4f}")
